# Audit Tasks
> Project health checks and symbol analysis

In [ ]:
#| default_exp tasks.audit

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, List, Optional, Set
from dataclasses import dataclass, field, asdict
from pathlib import Path
from collections import defaultdict
import asyncio

from mcp.server.fastmcp import FastMCP

from nbdev_mcp.utils.paths import (
    resolve_selector, iter_notebooks, read_nb, settings_dict, nbs_dir
)
from nbdev_mcp.utils.nb import (
    join_source, find_default_exp, extract_symbols, extract_imports
)
from nbdev_mcp.tasks.core import (
    NbdevTaskContext, format_task_result
)
from nbdev_mcp.tasks.cache import (
    ModidxCacheManager, SymbolStats
)

## Audit Result Structures

In [ ]:
#| export
@dataclass
class StructureIssue:
    """A structural issue in the project."""
    severity: str  # 'error', 'warning', 'info'
    category: str  # 'naming', 'export', 'import', 'structure'
    notebook: str
    message: str
    suggestion: Optional[str] = None


@dataclass
class AuditResult:
    """Complete audit result."""
    project: str
    notebook_count: int = 0
    symbol_count: int = 0
    issues: List[StructureIssue] = field(default_factory=list)
    coverage: Dict[str, float] = field(default_factory=dict)
    summary: Dict[str, int] = field(default_factory=dict)
    
    def to_dict(self) -> Dict[str, Any]:
        return {
            'project': self.project,
            'notebook_count': self.notebook_count,
            'symbol_count': self.symbol_count,
            'issues': [asdict(i) for i in self.issues],
            'coverage': self.coverage,
            'summary': self.summary
        }

## Audit Functions

In [ ]:
#| export
def check_notebook_structure(nb_path: Path, project: Path) -> List[StructureIssue]:
    """Check a single notebook for structural issues.
    
    Parameters
    ----------
    nb_path : Path
        Path to notebook.
    project : Path
        Project root.
    
    Returns
    -------
    List[StructureIssue]
        Issues found.
    """
    issues = []
    nb_name = str(nb_path.relative_to(project))
    
    try:
        nb_data = read_nb(nb_path)
    except Exception as e:
        issues.append(StructureIssue(
            severity='error',
            category='structure',
            notebook=nb_name,
            message=f'Failed to read notebook: {e}'
        ))
        return issues
    
    cells = nb_data.get('cells', [])
    
    # Check for default_exp
    default_exp = find_default_exp(nb_data)
    if not default_exp:
        issues.append(StructureIssue(
            severity='warning',
            category='export',
            notebook=nb_name,
            message='No #| default_exp directive found',
            suggestion='Add #| default_exp <module_name> to first code cell'
        ))
    
    # Check naming conventions
    fname = nb_path.stem
    if not fname[0].isdigit() and fname != 'index':
        issues.append(StructureIssue(
            severity='info',
            category='naming',
            notebook=nb_name,
            message='Notebook name should start with a number for ordering',
            suggestion=f'Rename to NN_{fname}.ipynb'
        ))
    
    # Check for __all__ definitions (bad practice in nbdev)
    for i, cell in enumerate(cells):
        if cell.get('cell_type') != 'code':
            continue
        src = join_source(cell.get('source', []))
        if '__all__' in src and '=' in src:
            issues.append(StructureIssue(
                severity='warning',
                category='export',
                notebook=nb_name,
                message=f'Cell {i}: __all__ definition found - nbdev handles exports automatically',
                suggestion='Remove __all__ and use #| export directive'
            ))
    
    # Check for relative imports in export cells
    for i, cell in enumerate(cells):
        if cell.get('cell_type') != 'code':
            continue
        src = join_source(cell.get('source', []))
        if '#| export' in src or '#|export' in src:
            if 'from .' in src or 'import .' in src:
                issues.append(StructureIssue(
                    severity='warning',
                    category='import',
                    notebook=nb_name,
                    message=f'Cell {i}: Relative import in export cell',
                    suggestion='Use absolute imports in export cells'
                ))
    
    return issues


def check_export_consistency(project: Path) -> List[StructureIssue]:
    """Check that exports are consistent across notebooks.
    
    Looks for:
    - Same symbol exported from multiple notebooks
    - Missing nbdev_export calls
    """
    issues = []
    symbol_sources: Dict[str, List[str]] = defaultdict(list)
    
    for nb_path in iter_notebooks(project):
        nb_name = str(nb_path.relative_to(project))
        nb_data = read_nb(nb_path)
        module = find_default_exp(nb_data) or ''
        
        has_exports = False
        has_export_call = False
        
        for cell in nb_data.get('cells', []):
            if cell.get('cell_type') != 'code':
                continue
            src = join_source(cell.get('source', []))
            
            if '#| export' in src or '#|export' in src:
                has_exports = True
                symbols = extract_symbols(src)
                for sym in symbols:
                    fqn = f"{module}.{sym.name}" if module else sym.name
                    symbol_sources[fqn].append(nb_name)
            
            if 'nbdev_export' in src or 'nbdev.nbdev_export' in src:
                has_export_call = True
        
        # Check for missing export call
        if has_exports and not has_export_call:
            issues.append(StructureIssue(
                severity='info',
                category='export',
                notebook=nb_name,
                message='Has exports but no nbdev.nbdev_export call',
                suggestion='Add nbdev.nbdev_export at the end'
            ))
    
    # Check for duplicate exports
    for fqn, sources in symbol_sources.items():
        if len(sources) > 1:
            issues.append(StructureIssue(
                severity='error',
                category='export',
                notebook=sources[0],
                message=f"Symbol '{fqn}' exported from multiple notebooks: {', '.join(sources)}",
                suggestion='Keep symbol in one notebook only'
            ))
    
    return issues


def check_import_health(project: Path) -> List[StructureIssue]:
    """Check import patterns for issues.
    
    Looks for:
    - Circular imports (potential)
    - Unused imports in export cells
    - Missing imports
    """
    issues = []
    settings = settings_dict(project)
    lib_name = settings.get('lib_name', 'pkg').replace('-', '_')
    
    # Build import graph
    imports_from: Dict[str, Set[str]] = defaultdict(set)  # module -> imports_these_modules
    
    for nb_path in iter_notebooks(project):
        nb_name = str(nb_path.relative_to(project))
        nb_data = read_nb(nb_path)
        module = find_default_exp(nb_data) or nb_name
        
        for cell in nb_data.get('cells', []):
            if cell.get('cell_type') != 'code':
                continue
            src = join_source(cell.get('source', []))
            
            imports = extract_imports(src)
            for imp in imports:
                if imp.module.startswith(lib_name):
                    imported_module = imp.module.replace(lib_name + '.', '')
                    imports_from[module].add(imported_module)
    
    # Simple cycle detection (direct cycles only)
    for module, imported in imports_from.items():
        for imp_mod in imported:
            if module in imports_from.get(imp_mod, set()):
                issues.append(StructureIssue(
                    severity='warning',
                    category='import',
                    notebook=module,
                    message=f'Potential circular import between {module} and {imp_mod}',
                    suggestion='Refactor to break the cycle'
                ))
    
    return issues


def compute_coverage(project: Path) -> Dict[str, float]:
    """Compute documentation and test coverage.
    
    Returns
    -------
    Dict[str, float]
        Coverage percentages.
    """
    total_symbols = 0
    documented_symbols = 0
    tested_cells = 0
    total_test_cells = 0
    
    for nb_path in iter_notebooks(project):
        nb_data = read_nb(nb_path)
        
        for cell in nb_data.get('cells', []):
            if cell.get('cell_type') != 'code':
                continue
            src = join_source(cell.get('source', []))
            
            # Check exports for docstrings
            if '#| export' in src or '#|export' in src:
                symbols = extract_symbols(src)
                for sym in symbols:
                    total_symbols += 1
                    # Simple docstring check
                    if '"""' in src or "'''" in src:
                        documented_symbols += 1
            
            # Check for test cells
            if 'assert' in src or 'test_' in src:
                total_test_cells += 1
                if '#| eval: false' not in src.lower():
                    tested_cells += 1
    
    return {
        'documentation': (documented_symbols / max(1, total_symbols)) * 100,
        'active_tests': (tested_cells / max(1, total_test_cells)) * 100 if total_test_cells > 0 else 0
    }

## Full Project Audit

In [ ]:
#| export
async def full_project_audit(
    project: Optional[str] = None,
    ctx: Optional[NbdevTaskContext] = None
) -> AuditResult:
    """Run comprehensive project audit.
    
    Checks:
    - Notebook structure and naming
    - Export consistency
    - Import health
    - Documentation coverage
    - Symbol usage
    
    Parameters
    ----------
    project : str, optional
        Project path or alias.
    ctx : NbdevTaskContext, optional
        Task context for progress updates.
    
    Returns
    -------
    AuditResult
        Complete audit result.
    """
    p = resolve_selector(project)
    result = AuditResult(project=str(p))
    
    notebooks = list(iter_notebooks(p))
    result.notebook_count = len(notebooks)
    
    if ctx:
        await ctx.update(1, 'Checking notebook structure...')
    
    # Step 1: Structure checks
    for nb_path in notebooks:
        issues = check_notebook_structure(nb_path, p)
        result.issues.extend(issues)
    
    if ctx:
        await ctx.update(2, 'Checking export consistency...')
    
    # Step 2: Export consistency
    result.issues.extend(check_export_consistency(p))
    
    if ctx:
        await ctx.update(3, 'Analyzing imports...')
    
    # Step 3: Import health
    result.issues.extend(check_import_health(p))
    
    if ctx:
        await ctx.update(4, 'Computing coverage...')
    
    # Step 4: Coverage
    result.coverage = compute_coverage(p)
    
    if ctx:
        await ctx.update(5, 'Gathering symbol stats...')
    
    # Step 5: Symbol stats from cache
    cache = ModidxCacheManager.get(p)
    result.symbol_count = len(cache.symbol_stats)
    
    # Count orphans as potential issues
    orphan_count = sum(
        1 for s in cache.symbol_stats.values()
        if s.import_count == 0 and s.line_count >= 5
    )
    if orphan_count > 0:
        result.issues.append(StructureIssue(
            severity='info',
            category='structure',
            notebook='project',
            message=(
                f'{orphan_count} symbols with 5+ lines have zero imports '
                '(potential dead code; weigh by module depth and tutorial usage; may be public API)'
            ),
            suggestion='Run orphan_symbols tool for details'
        ))
    
    # Summary
    result.summary = {
        'errors': sum(1 for i in result.issues if i.severity == 'error'),
        'warnings': sum(1 for i in result.issues if i.severity == 'warning'),
        'info': sum(1 for i in result.issues if i.severity == 'info')
    }
    
    return result


## Symbol Audit

In [ ]:
#| export
def symbol_audit(
    project: Optional[str] = None,
    symbol: Optional[str] = None
) -> Dict[str, Any]:
    """Audit a specific symbol.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias.
    symbol : str, optional
        Symbol FQN to audit.
    
    Returns
    -------
    Dict[str, Any]
        Symbol audit details.
    """
    p = resolve_selector(project)
    cache = ModidxCacheManager.get(p)
    
    if symbol not in cache.symbol_stats:
        return {'ok': False, 'error': f'Symbol not found: {symbol}'}
    
    stats = cache.symbol_stats[symbol]
    graph = cache.dependency_graph
    
    # Find dependents and dependencies
    dependents = []
    dependencies = []
    
    if graph:
        for edge in graph.edges:
            if edge.source == symbol:
                dependents.append({
                    'target': edge.target,
                    'kind': edge.kind
                })
            if edge.target == symbol:
                dependencies.append({
                    'source': edge.source,
                    'kind': edge.kind
                })
    
    # Get symbol node for more details
    sym_node = graph.symbols.get(symbol) if graph else None
    
    return {
        'ok': True,
        'symbol': symbol,
        'stats': stats.to_dict(),
        'location': {
            'notebook': sym_node.notebook if sym_node else None,
            'cell_index': sym_node.cell_index if sym_node else None
        },
        'dependents': dependents,
        'dependencies': dependencies,
        'health': {
            'is_orphan': stats.import_count == 0,
            'is_hot': stats.import_count >= 5,
            'is_large': stats.line_count >= 50
        }
    }

## MCP Tool Registration

In [ ]:
#| export
def add_audit_tools(mcp: FastMCP) -> None:
    """Register audit tools with MCP server.
    
    Parameters
    ----------
    mcp : FastMCP
        The MCP server instance.
    """
    
    @mcp.tool()
    async def project_audit(
        project: Optional[str] = None
    ) -> Dict[str, Any]:
        """Run comprehensive project health audit.
        
        Checks notebook structure, exports, imports, and coverage.
        Returns issues grouped by severity.
        """
        ctx = NbdevTaskContext(
            project=resolve_selector(project),
            total_steps=5
        )
        result = await full_project_audit(project, ctx)
        
        return {
            'ok': True,
            **result.to_dict()
        }
    
    @mcp.tool()
    def audit_symbol(
        project: Optional[str] = None,
        symbol: str = ''
    ) -> Dict[str, Any]:
        """Audit a specific symbol.
        
        Shows usage stats, dependents, dependencies, and health indicators.
        """
        return symbol_audit(project, symbol)
    
    @mcp.tool()
    def check_notebook(
        project: Optional[str] = None,
        notebook: str = ''
    ) -> Dict[str, Any]:
        """Check a specific notebook for issues.
        
        Validates structure, exports, and imports.
        """
        p = resolve_selector(project)
        nb_path = p / notebook
        
        if not nb_path.exists():
            # Try in nbs directory
            nb_path = nbs_dir(p) / notebook
        
        if not nb_path.exists():
            return {'ok': False, 'error': f'Notebook not found: {notebook}'}
        
        issues = check_notebook_structure(nb_path, p)
        
        return {
            'ok': True,
            'notebook': notebook,
            'issues': [asdict(i) for i in issues],
            'issue_count': len(issues)
        }

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()